# 📚 Image Processing with OpenCV — Learning Notes
### Day 03 | Computer Vision | Lalana Gurusinghe

---

# 📋 Table of Contents

1. [What is Image Processing?](#1-what-is-image-processing)
2. [Point Operations](#2-point-operations)
3. [Color Spaces](#3-color-spaces)
4. [HSV Color Masking](#4-hsv-color-masking)
5. [Thresholding](#5-thresholding)
6. [Contour Detection](#6-contour-detection)
7. [Plant Height Measurement Project](#7-plant-height-measurement-project)

---

# 1. What is Image Processing?

Image processing is the technique of performing operations on an image to:
- **Enhance** it (brightness, contrast)
- **Extract information** from it (colors, shapes, measurements)
- **Transform** it (resize, rotate, convert)

### How a Digital Image Works

Every image is just a **grid of pixels**. Each pixel has a numerical value representing its color or brightness.

```
Grayscale Image (each pixel = 1 number, 0–255):

  0    0    0    0    0
  0  255  255  255    0
  0  255  255  255    0
  0  255  255  255    0
  0    0    0    0    0

  0   = black
  255 = white
  128 = gray
```

```
Color Image (each pixel = 3 numbers, BGR):

  pixel = (Blue, Green, Red)
  
  Pure Red   = (0,   0,   255)
  Pure Green = (0,   255, 0)
  Pure Blue  = (255, 0,   0)
  White      = (255, 255, 255)
  Black      = (0,   0,   0)
```

### Image Shape in OpenCV

```python
import cv2

img = cv2.imread('image.jpg')
print(img.shape)
# Output: (height, width, channels)
# Example: (1080, 1920, 3)
#            H     W    C
#           [0]   [1]  [2]
```

| Index | Value | Meaning |
|-------|-------|---------|
| `[0]` | e.g. 1080 | Height — rows from top to bottom |
| `[1]` | e.g. 1920 | Width — columns left to right |
| `[2]` | 3 | Channels — BGR (Blue, Green, Red) |

> ⚠️ `shape` is a **property** not a function — never use `shape()` with brackets

```python
# ❌ Wrong
print(img.shape())

# ✅ Correct
print(img.shape)
```

### Channel Types

| Channels | Format | shape value | Used For |
|----------|--------|-------------|----------|
| 1 | Grayscale | `(h, w)` | Black & white |
| 3 | BGR | `(h, w, 3)` | Normal color images |
| 4 | BGRA | `(h, w, 4)` | PNG with transparency |

---

# 2. Point Operations

A **point operation** transforms each pixel independently based only on its own value — not its neighbors.

$$
g(x, y) = T[f(x, y)]
$$

Where:
- $f(x, y)$ = input pixel value at position (x, y)
- $g(x, y)$ = output pixel value
- $T$ = the transformation function

---

## 2.1 Image Negative

Inverts the image — dark becomes bright, bright becomes dark.

$$
g(x, y) = 255 - f(x, y)
$$

```python
import cv2

img = cv2.imread('image.jpg', 0)  # 0 = load as grayscale
negative = 255 - img
# Every pixel is subtracted from 255
# pixel 0   (black) → 255 (white)
# pixel 255 (white) → 0   (black)
# pixel 100 (gray)  → 155 (lighter gray)

cv2.imshow('Negative Image', negative)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

## 2.2 Brightness Adjustment

Add or subtract a constant from every pixel.

$$
g(x, y) = f(x, y) + C
$$

- `C > 0` → brighter
- `C < 0` → darker

```python
bright = cv2.add(img, 50)
# cv2.add() is used instead of img + 50
# because cv2.add() handles overflow safely
# (clips at 255 instead of wrapping around)

# Example:
# pixel 200 + 50 = 250  ✅ normal
# pixel 220 + 50 = 255  ✅ clipped at 255 (safe)
# pixel 220 + 50 = 14   ❌ wrong (overflow without cv2.add)

cv2.imshow('Brightness Increased', bright)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

## 2.3 Contrast Adjustment

Multiply each pixel by a constant.

$$
g(x, y) = C \cdot f(x, y)
$$

- `C > 1` → higher contrast (brighter brights, darker darks)
- `0 < C < 1` → lower contrast (everything becomes more gray)

```python
contrast = cv2.convertScaleAbs(img, alpha=1.5, beta=0)
# alpha = contrast multiplier (C in the formula)
# beta  = brightness offset (like brightness adjustment)
# convertScaleAbs handles clipping at 0 and 255

cv2.imshow('Contrast Adjusted', contrast)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

## 2.4 Log Transformation

Enhances dark regions while compressing bright regions. Useful for images with wide intensity ranges.

$$
g(x, y) = C \log(1 + f(x, y))
$$

- The `+1` prevents log(0) which is undefined
- `C` is calculated to stretch the result back to 0–255 range

```python
import numpy as np
import cv2

img = cv2.imread('image.jpg', 0)

c = 255 / np.log(1 + np.max(img))
# c scales the result so the maximum value maps to 255

log_image = c * (np.log(1 + img))
# Apply log to every pixel

log_image = np.array(log_image, dtype=np.uint8)
# Convert back to uint8 (0-255 integer format)
# np.uint8 NOT np.unit8 (common typo!)

cv2.imshow('Log Transformed', log_image)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

## 2.5 Power Law (Gamma) Transformation

Used for gamma correction — adjusts brightness non-linearly.

$$
g(x, y) = C \cdot f(x, y)^\gamma
$$

- `γ < 1` → brighter image (enhances dark areas)
- `γ > 1` → darker image (enhances bright areas)
- `γ = 1` → no change

```python
import cv2
import numpy as np

img = cv2.imread('image.jpg', 0)

gamma = 0.5
# γ = 0.5 makes the image brighter
# γ = 2.0 makes the image darker

gamma_corrected = cv2.pow(img / 255.0, gamma) * 255
# img/255.0 → normalize to 0.0–1.0 range
# cv2.pow()  → apply the power (gamma)
# * 255      → scale back to 0–255

gamma_corrected = np.array(gamma_corrected, dtype=np.uint8)

cv2.imshow('Gamma Corrected', gamma_corrected)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

## 2.6 Custom Grayscale Conversion

Building your own BGR → Grayscale converter using NumPy:

```python
import cv2
import numpy as np

def myCvtColor(img):
    img_new = np.mean(img, axis=2)
    # axis=2 means average across the 3 color channels (B, G, R)
    # Each pixel's 3 values (B,G,R) are averaged into 1 value
    # Shape changes: (h, w, 3) → (h, w)

    img_new = np.array(img_new, dtype=np.uint8)
    # np.uint8 = unsigned integer 8-bit = values 0 to 255
    # This is the standard format for OpenCV images

    print(img.shape, img_new.shape)
    # Before: (480, 640, 3)   ← color image
    # After:  (480, 640)      ← grayscale image

    return img_new

img = cv2.imread('image.jpg')
gray = myCvtColor(img)
```

> 💡 OpenCV's built-in `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` uses a weighted formula instead of a simple average — it gives more weight to green because human eyes are more sensitive to green.

---

# 3. Color Spaces

A **color space** is a system for representing colors as numbers.

## 3.1 BGR (OpenCV Default)

```
pixel = (Blue, Green, Red)

Pure Red   = (0,   0,   255)
Pure Green = (0,   255, 0  )
Pure Blue  = (255, 0,   0  )
```

> ⚠️ OpenCV uses **BGR** not RGB — the order is reversed from most other libraries

## 3.2 HSV (Best for Color Detection)

```
pixel = (Hue, Saturation, Value)
```

| Channel | Meaning | Range in OpenCV |
|---------|---------|-----------------|
| **H** (Hue) | The actual color (red, green, blue...) | 0 – 179 |
| **S** (Saturation) | How vivid/pure the color is | 0 – 255 |
| **V** (Value) | Brightness of the color | 0 – 255 |

```
Hue color wheel (OpenCV range 0–179):

  0/179 ── Red
    20  ── Orange/Yellow
    30  ── Yellow
    60  ── Yellow-Green
    75  ── Green
   100  ── Cyan
   120  ── Blue
   150  ── Magenta/Purple
```

### Why use HSV for color detection?

```
Detecting "green" in BGR:
  dark green  = (0, 100, 0)
  light green = (144, 238, 144)
  lime green  = (50, 205, 50)
  → Many different combinations → hard to define a range

Detecting "green" in HSV:
  dark green  = (75, 255, 100)
  light green = (75, 80, 238)
  lime green  = (73, 195, 205)
  → H is always around 75 → easy to define a range!
```

### Converting Between Color Spaces

```python
import cv2

img = cv2.imread('image.jpg')

# BGR → HSV  (most useful for color detection)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# BGR → Grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# BGR → RGB  (for matplotlib display)
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# BGR → LAB  (for perceptual color difference)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
```

---

# 4. HSV Color Masking

Color masking isolates a specific color range in an image, creating a **binary mask** where detected pixels are white and everything else is black.

## 4.1 How it Works

```
Original Image   →   HSV Conversion   →   inRange()   →   Mask
  🔴🟢🔵🟡              HSV values          threshold      ⬛⬜⬛⬛
  🟢🔴🟢🔵      →      for each pixel   →   check      →  ⬜⬛⬜⬛
  🔵🟢🔴🟢              compared to          range          ⬛⬜⬛⬜
                         our limits                         ⬜⬛⬜⬛
                                                    white = in range ✅
                                                    black = out of range ❌
```

## 4.2 Basic Color Masking Code

```python
import cv2
import numpy as np

# ── Load image ─────────────────────────────────────────────────────
frame = cv2.imread('image.jpg')
# Loads in BGR format

# ── Convert to HSV ─────────────────────────────────────────────────
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
# HSV is better for color detection (see section 3.2)

# ── Define color range ─────────────────────────────────────────────
lower_color = np.array((96, 227, 130))   # (H_min, S_min, V_min)
upper_color = np.array((140, 255, 255))  # (H_max, S_max, V_max)
# This range targets BLUE color
# H: 96-140  → blue hues
# S: 227-255 → very vivid blue only
# V: 130-255 → medium to bright blue only

# ── Create mask ────────────────────────────────────────────────────
mask = cv2.inRange(hsv, lower_color, upper_color)
# Scans every pixel:
#   within range  → 255 (white)
#   outside range → 0   (black)

# ── Apply mask ─────────────────────────────────────────────────────
res = cv2.bitwise_and(frame, frame, mask=mask)
# mask=255 → keep original pixel
# mask=0   → set pixel to black (0,0,0)
# Result: only blue pixels are visible

cv2.imshow('frame', frame)   # original
cv2.imshow('mask',  mask)    # binary mask
cv2.imshow('res',   res)     # isolated color

cv2.waitKey(0)
cv2.destroyAllWindows()
```

## 4.3 HSV Color Reference Table

| Color | H Lower | H Upper | S Lower | V Lower |
|-------|---------|---------|---------|---------|
| Red | 0 | 10 | 100 | 100 |
| Orange | 10 | 25 | 100 | 100 |
| Yellow | 20 | 35 | 100 | 100 |
| Green (plant) | 25 | 95 | 40 | 40 |
| Cyan | 80 | 100 | 100 | 100 |
| Blue | 96 | 140 | 100 | 100 |
| Purple | 125 | 155 | 100 | 100 |
| White | 0 | 180 | 0 | 200 |
| Black | 0 | 180 | 0 | 0 |

## 4.4 Interactive Color Picker using Trackbars

```python
import cv2
import numpy as np

def nothing(x):
    pass
# Callback function required by createTrackbar
# Does nothing — we read values manually in the loop

# Create window first, then attach trackbars to it
cv2.namedWindow('result')

cv2.createTrackbar('h', 'result', 0, 179, nothing)  # Hue:        0–179
cv2.createTrackbar('s', 'result', 0, 255, nothing)  # Saturation: 0–255
cv2.createTrackbar('v', 'result', 0, 255, nothing)  # Value:      0–255

while True:
    frame = cv2.imread('image.jpg')

    if frame is None:
        print("Image not found!")
        break

    frame = cv2.resize(frame, (650, 350))
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Read current trackbar positions
    h = cv2.getTrackbarPos('h', 'result')
    s = cv2.getTrackbarPos('s', 'result')
    v = cv2.getTrackbarPos('v', 'result')

    lower_color = np.array((h, s, v))
    upper_color = np.array((102, 255, 255))

    mask   = cv2.inRange(hsv, lower_color, upper_color)
    result = cv2.bitwise_and(frame, frame, mask=mask)

    cv2.imshow('result', result)

    # Check if window was closed
    if cv2.getWindowProperty('result', cv2.WND_PROP_VISIBLE) < 1:
        break

    k = cv2.waitKey(5) & 0xFF
    if k == 27:   # ESC key to exit
        break

cv2.destroyAllWindows()
```

> ⚠️ **Jupyter Notebook tip:** `cv2.imshow()` crashes the kernel in Jupyter. Use **Matplotlib** instead:

```python
import matplotlib.pyplot as plt

# Always convert BGR → RGB before showing in matplotlib
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

# For grayscale/mask images:
plt.imshow(mask, cmap='gray')
plt.show()
```

---

# 5. Thresholding

Thresholding converts a **grayscale image into a binary image** by comparing each pixel to a threshold value T.

$$
g(x, y) = \begin{cases} 255 & \text{if } f(x,y) > T \\ 0 & \text{otherwise} \end{cases}
$$

## 5.1 Basic Thresholding

```python
import cv2

img = cv2.imread('image.jpg', 0)  # load as grayscale

ret, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
# img   → input (must be grayscale)
# 127   → threshold value T
# 255   → value to set for pixels that pass the threshold
# ret   → the threshold value used (127 in this case)
# thresh → the resulting binary image

cv2.imshow('Original',    img)
cv2.imshow('Thresholded', thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

## 5.2 Types of Thresholding

### THRESH_BINARY (most common)

$$
g(x, y) = \begin{cases} 255 & \text{if } f(x,y) > T \\ 0 & \text{otherwise} \end{cases}
$$

```python
ret, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
# pixel > 127 → 255 (white)
# pixel ≤ 127 → 0   (black)
```

### THRESH_BINARY_INV (inverted)

```python
ret, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
# pixel > 127 → 0   (black)  ← opposite of BINARY
# pixel ≤ 127 → 255 (white)
```

### THRESH_TRUNC (truncate — ceiling)

$$
g(x, y) = \begin{cases} T & \text{if } f(x,y) > T \\ f(x,y) & \text{otherwise} \end{cases}
$$

Think of it as a **brightness ceiling** — pixels can't go above T.

```python
ret, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)
# pixel > 127 → capped at 127  (not allowed to be brighter)
# pixel ≤ 127 → stays unchanged

# Example with T=127:
# pixel 200 → becomes 127
# pixel 80  → stays 80
# pixel 50  → stays 50
```

### THRESH_TOZERO (zero out dark pixels)

$$
g(x, y) = \begin{cases} f(x,y) & \text{if } f(x,y) > T \\ 0 & \text{otherwise} \end{cases}
$$

Think of it as a **filter** — only bright pixels survive, dark ones become black.

```python
ret, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)
# pixel > 127 → stays unchanged
# pixel ≤ 127 → becomes 0 (black)

# Example with T=127:
# pixel 200 → stays 200
# pixel 80  → becomes 0
# pixel 50  → becomes 0
```

### Comparison Table

| Pixel Value | THRESH_BINARY | THRESH_TRUNC | THRESH_TOZERO |
|-------------|---------------|--------------|---------------|
| 200 | 255 | 127 (capped) | 200 (unchanged) |
| 150 | 255 | 127 (capped) | 150 (unchanged) |
| 80  | 0   | 80 (unchanged) | 0 (wiped) |
| 50  | 0   | 50 (unchanged) | 0 (wiped) |

### THRESH_OTSU (automatic threshold)

Automatically finds the best threshold value — no need to manually set T.

```python
ret, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# 0 is passed as threshold because Otsu calculates it automatically
# ret now contains the calculated threshold value
print("Optimal threshold:", ret)
```

---

# 6. Contour Detection

A **contour** is the boundary/outline of an object in an image.

```
Original Image        Contour
┌──────────┐          ┌──────────┐
│          │          │          │
│  ██████  │   →      │  ┌────┐  │
│  ██████  │          │  │    │  │
│  ██████  │          │  └────┘  │
│          │          │          │
└──────────┘          └──────────┘
  filled shape          only the edge
```

## 6.1 Finding Contours

```python
import cv2
import numpy as np

img    = cv2.imread('image.jpg')
gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Contours work on grayscale or binary images, not color

ret, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
# Convert to binary — findContours needs a black/white image

contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
# contours  → list of all detected contour point arrays
# hierarchy → parent/child relationship between contours
```

### cv2.findContours() Parameters

**Parameter 2 — Retrieval Mode:**

| Mode | What it finds |
|------|--------------|
| `RETR_EXTERNAL` | Only outermost contours (ignores holes inside shapes) |
| `RETR_LIST` | All contours, no hierarchy info |
| `RETR_TREE` | All contours + full parent/child tree |

**Parameter 3 — Approximation Method:**

| Method | Stores | Memory |
|--------|--------|--------|
| `CHAIN_APPROX_NONE` | Every single point on boundary | More |
| `CHAIN_APPROX_SIMPLE` | Only corner/endpoint points | Less ✅ |

```
CHAIN_APPROX_NONE          CHAIN_APPROX_SIMPLE
● ● ● ● ● ● ●              ●             ●
●           ●
●           ●
● ● ● ● ● ● ●              ●             ●
Stores 16 points            Stores 4 corners only
```

## 6.2 Drawing Contours

```python
cv2.drawContours(img, contours, -1, (0, 255, 0), 2)
# img       → image to draw on
# contours  → list of contours
# -1        → draw ALL contours (-1 = all, 0/1/2 = specific index)
# (0,255,0) → color in BGR (green)
# 2         → line thickness in pixels

# Draw only ONE specific contour (e.g. index 8):
cv2.drawContours(img, contours, 8, (0, 255, 255), 2)

# Draw a contour from the loop:
cv2.drawContours(img, [cnt], -1, (0, 255, 255), 1)
# [cnt] wraps single contour in a list
```

## 6.3 Contour Features

```python
for cnt in contours:

    # ── Area ───────────────────────────────────────────────────────
    area = cv2.contourArea(cnt)
    # Returns area in pixels²
    # Useful for filtering out small noise contours
    if area > 1200.0:   # only process contours larger than 1200 pixels
        pass

    # ── Perimeter ──────────────────────────────────────────────────
    perimeter = cv2.arcLength(cnt, True)
    # True  = closed contour (shape)
    # False = open contour (curve/line)

    # ── Bounding Rectangle ─────────────────────────────────────────
    x, y, w, h = cv2.boundingRect(cnt)
    # x, y → top-left corner of the rectangle
    # w    → width  of the rectangle in pixels
    # h    → height of the rectangle in pixels
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # ── Centroid (center point) ────────────────────────────────────
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])   # center x
        cy = int(M['m01'] / M['m00'])   # center y
        cv2.circle(img, (cx, cy), 5, (0, 0, 255), -1)
```

## 6.4 Hierarchy

Describes the **nesting relationship** between contours:

```
Outer shape ──── parent
    └── Inner hole ──── child
            └── Shape inside hole ──── grandchild
```

```python
# hierarchy[0][i] = [Next, Previous, FirstChild, Parent]
# -1 means "does not exist"

for i, cnt in enumerate(contours):
    next_c, prev_c, child_c, parent_c = hierarchy[0][i]
    print(f"Contour {i}: parent={parent_c}, child={child_c}")
```

---

# 7. Plant Height Measurement Project

## 7.1 Project Goal

Detect green plant regions in images and measure their real-world height in centimeters using a reference scale.

## 7.2 Theory — Pixel to Real-World Conversion

To convert pixel measurements to real-world units, we need a **reference object** with a known real size placed in the image.

$$
\text{pixels per cm} = \frac{\text{Reference Pixels}}{\text{Reference Real CM}}
$$

$$
\text{Real Height (cm)} = \frac{\text{Plant Height (pixels)}}{\text{pixels per cm}}
$$

**Example:**
- Credit card height = 5.4 cm
- Credit card in image = 150 pixels
- pixels per cm = 150 / 5.4 = **27.78 px/cm**
- Plant height = 344 pixels
- Real height = 344 / 27.78 = **12.38 cm**

## 7.3 Full Project Code with Explanation

```python
import cv2
import numpy as np

# ══════════════════════════════════════════════════════════════════
# REFERENCE SCALE SETUP
# Place a known object next to the plant in the photo
# Measure its pixel height from the image
# ══════════════════════════════════════════════════════════════════
REFERENCE_REAL_CM = 5.4    # credit card real height in cm
REFERENCE_PIXELS  = 150    # credit card height in pixels (measure manually)
pixels_per_cm     = REFERENCE_PIXELS / REFERENCE_REAL_CM
# pixels_per_cm = 150 / 5.4 = 27.78 pixels per 1 cm

# ══════════════════════════════════════════════════════════════════
# STEP 1 — Load Image
# ══════════════════════════════════════════════════════════════════
plant_img = cv2.imread(r'plants/D10.png')
# imread returns None if file not found — always check!
if plant_img is None:
    print("Image not found!")

# ══════════════════════════════════════════════════════════════════
# STEP 2 — Resize
# ══════════════════════════════════════════════════════════════════
height, width = plant_img.shape[0:2]
# shape[0] = height, shape[1] = width, shape[2] = channels (ignored)

plant_img = cv2.resize(plant_img, (int(width/2), int(height/2)))
# cv2.resize takes (WIDTH, HEIGHT) — not (height, width)
# Resize to 50% of original for faster processing

# ══════════════════════════════════════════════════════════════════
# STEP 3 — Convert BGR to HSV
# ══════════════════════════════════════════════════════════════════
hsv = cv2.cvtColor(plant_img, cv2.COLOR_BGR2HSV)
# HSV makes it easy to define a color range for green

# ══════════════════════════════════════════════════════════════════
# STEP 4 — Define Green Color Range
# ══════════════════════════════════════════════════════════════════
lower_color_limit = np.array([25,  40,  40])
upper_color_limit = np.array([95, 255, 255])
#                              H    S    V
#
# H: 25–95   → covers yellow-green to dark green
# S: 40–255  → moderately vivid to fully vivid (avoids dull grays)
# V: 40–255  → medium to full brightness (avoids shadows/black)

# ══════════════════════════════════════════════════════════════════
# STEP 5 — Create Binary Mask
# ══════════════════════════════════════════════════════════════════
mask = cv2.inRange(hsv, lower_color_limit, upper_color_limit)
# For every pixel in hsv:
#   within [lower, upper] range → 255 (white) = plant detected
#   outside range               → 0   (black) = not plant

# ══════════════════════════════════════════════════════════════════
# STEP 6 — Apply Mask (isolate green regions)
# ══════════════════════════════════════════════════════════════════
res = cv2.bitwise_and(plant_img, plant_img, mask=mask)
# mask pixel = 255 → keep original color
# mask pixel = 0   → set to black (0,0,0)
# Result: only green plant areas visible

# ══════════════════════════════════════════════════════════════════
# STEP 7 — Find Contours
# ══════════════════════════════════════════════════════════════════
contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
print('no of Contours :', len(contours))
# contours = list of boundary point arrays for each detected region
# Many small contours may be noise — we filter by area next

# ══════════════════════════════════════════════════════════════════
# STEP 8 — Process Each Contour
# ══════════════════════════════════════════════════════════════════
for cnt in contours:
    area = cv2.contourArea(cnt)
    print(area)

    if area > 1200.0:
        # ── Filter: only process contours larger than 1200 px² ─────
        # Removes noise, dust, small reflections

        # ── Draw contour outline in yellow ─────────────────────────
        cv2.drawContours(plant_img, [cnt], -1, (0, 255, 255), 1)

        # ── Get bounding rectangle ──────────────────────────────────
        x, y, w, h = cv2.boundingRect(cnt)
        # x, y = top-left corner position
        # w    = width  of rectangle in pixels
        # h    = height of rectangle in pixels ← we use this for measurement

        # ── Draw bounding rectangle in green ───────────────────────
        cv2.rectangle(plant_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

        # ── Draw measurement circles ────────────────────────────────
        cv2.circle(plant_img, (int(x+w/2), y),   7, (0, 162, 255), -1)
        # Top circle at (center-x, top-y) of bounding box

        cv2.circle(plant_img, (int(x+w/2), y+h), 7, (0, 162, 255), -1)
        # Bottom circle at (center-x, bottom-y) of bounding box
        # -1 = filled circle

        # ── Convert pixel height to real cm ────────────────────────
        real_height_cm = round(h / pixels_per_cm, 2)
        # h / pixels_per_cm → converts pixels to cm
        # round(..., 2)     → keeps 2 decimal places

        # ── Display height on image ─────────────────────────────────
        cv2.putText(plant_img,
                    "Height: " + str(real_height_cm) + "cm",
                    # text — combine label + value into ONE string
                    (x+w, y+h),
                    # position — bottom-right of bounding box
                    cv2.FONT_HERSHEY_SIMPLEX,
                    # font type
                    1,
                    # font scale (size)
                    (255, 255, 255),
                    # color — white in BGR
                    2)
                    # thickness

# ══════════════════════════════════════════════════════════════════
# STEP 9 — Display Results
# ══════════════════════════════════════════════════════════════════
cv2.imshow('Plant Image', plant_img)  # image with all drawings
cv2.imshow('Mask', mask)              # binary mask
cv2.waitKey(0)         # wait until any key is pressed
cv2.destroyAllWindows() # close all OpenCV windows
```

## 7.4 Jupyter-Safe Version (with Matplotlib)

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

REFERENCE_REAL_CM = 5.4
REFERENCE_PIXELS  = 150
pixels_per_cm     = REFERENCE_PIXELS / REFERENCE_REAL_CM

plant_img = cv2.imread(r'plants/D10.png')
height, width = plant_img.shape[0:2]
plant_img = cv2.resize(plant_img, (int(width/2), int(height/2)))
original  = plant_img.copy()   # save clean copy for display

hsv = cv2.cvtColor(plant_img, cv2.COLOR_BGR2HSV)

lower_color_limit = np.array([25,  40,  40])
upper_color_limit = np.array([95, 255, 255])

mask = cv2.inRange(hsv, lower_color_limit, upper_color_limit)
res  = cv2.bitwise_and(plant_img, plant_img, mask=mask)

contours, hierarchy = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area > 1200.0:
        cv2.drawContours(plant_img, [cnt], -1, (0, 255, 255), 1)
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(plant_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.circle(plant_img, (int(x+w/2), y),   7, (0, 162, 255), -1)
        cv2.circle(plant_img, (int(x+w/2), y+h), 7, (0, 162, 255), -1)
        real_height_cm = round(h / pixels_per_cm, 2)
        cv2.putText(plant_img, "Height: " + str(real_height_cm) + "cm",
                    (x+w, y+h), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

# Display with matplotlib (Jupyter safe — no kernel crash)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(original,   cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Green Mask')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(plant_img,  cv2.COLOR_BGR2RGB))
axes[2].set_title('Measurements')
axes[2].axis('off')

plt.tight_layout()
plt.show()
```

## 7.5 Project Pipeline Summary

```
Input: Plant photo (D10.png)
         ↓
    Resize to 50%
    (faster processing)
         ↓
    BGR → HSV
    (better for color detection)
         ↓
    inRange([25,40,40], [95,255,255])
    (create binary mask of green regions)
         ↓
    bitwise_and(image, image, mask)
    (isolate green areas)
         ↓
    findContours(mask, RETR_TREE, CHAIN_APPROX_SIMPLE)
    (find boundaries of green blobs)
         ↓
    Filter: area > 1200 pixels
    (remove noise)
         ↓
    boundingRect(cnt) → x, y, w, h
    (get position and size)
         ↓
    real_height_cm = h / pixels_per_cm
    (convert to real world cm)
         ↓
    Draw: contour + rectangle + circles + text
         ↓
Output: Image with plant measurements in cm
```

---

# 📌 Common Errors & Fixes

| Error | Cause | Fix |
|-------|-------|-----|
| `TypeError: unsupported operand '-': NoneType` | Image not loaded (wrong path) | Check file path and extension |
| `AttributeError: module 'cv2' has no attribute 'cvtcolor'` | Wrong capitalization | Use `cvtColor` (capital C) |
| `AttributeError: module 'numpy' has no attribute 'unit8'` | Typo | Use `uint8` not `unit8` |
| `error: NULL window` | Window closed before trackbar read | Add `getWindowProperty` check |
| `error: !ssize.empty() in resize` | Image is None before resize | Check imread returned valid image |
| `error: Bad argument in putText` | Passing extra argument | Combine text + value into ONE string |
| Kernel restarts in Jupyter | `cv2.imshow()` not supported | Use `matplotlib.pyplot` instead |

---

# 📌 Quick Reference Cheatsheet

```python
# ── Load ────────────────────────────────────────────────────────────
img  = cv2.imread('file.jpg')          # color (BGR)
img  = cv2.imread('file.jpg', 0)       # grayscale
img  = cv2.imread('file.jpg', cv2.IMREAD_UNCHANGED)  # with alpha

# ── Info ────────────────────────────────────────────────────────────
img.shape        # (height, width, channels)
img.dtype        # data type (uint8)
img.size         # total number of pixels

# ── Convert ─────────────────────────────────────────────────────────
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ── Resize ──────────────────────────────────────────────────────────
resized = cv2.resize(img, (width, height))   # always (W, H)

# ── Threshold ───────────────────────────────────────────────────────
ret, th = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
ret, th = cv2.threshold(gray, 127, 255, cv2.THRESH_TRUNC)
ret, th = cv2.threshold(gray, 127, 255, cv2.THRESH_TOZERO)
ret, th = cv2.threshold(gray, 0,   255, cv2.THRESH_OTSU)

# ── Color Mask ──────────────────────────────────────────────────────
mask = cv2.inRange(hsv, lower, upper)
res  = cv2.bitwise_and(img, img, mask=mask)

# ── Contours ────────────────────────────────────────────────────────
cnts, hier = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img, cnts, -1, (0,255,0), 2)
area  = cv2.contourArea(cnt)
x,y,w,h = cv2.boundingRect(cnt)

# ── Draw ────────────────────────────────────────────────────────────
cv2.rectangle(img, (x,y), (x+w,y+h), (0,255,0), 2)
cv2.circle(img, (cx,cy), radius, (0,0,255), -1)   # -1 = filled
cv2.putText(img, "text", (x,y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
cv2.line(img, (x1,y1), (x2,y2), (255,0,0), 2)

# ── Display ─────────────────────────────────────────────────────────
cv2.imshow('window', img)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

---

*Notes compiled from Day 03 Computer Vision module — Lalana Gurusinghe*